# 05 - Borrower segmentation

Unsupervised segmentation of the booked portfolio using only
origination-time variables. We compare k-means (with the number of clusters
chosen by the elbow/silhouette criterion) and Ward hierarchical clustering,
project the segments on the first two principal components, and profile each
segment by size and realised default rate.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / 'config' / 'config.yaml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 180)

In [ ]:
from src.config import Config
from src.pipeline import load_modeling_frame, CLUSTER_FEATURES
from src import clustering

cfg = Config.load()
df = load_modeling_frame(cfg)
out = clustering.run_clustering(df, CLUSTER_FEATURES, k=None, k_range=range(2, 9), seed=cfg.seed, max_rows=60_000)
print('chosen k =', out['k'])

In [ ]:
display(out['k_selection'].round(4))
display(out['kmeans_profile'].round(4))

In [ ]:
display(out['hierarchical_profile'].round(4))

In [ ]:
for name in ['clustering_elbow.png', 'clustering_pca.png', 'clustering_dendrogram.png']:
    p = ROOT / 'reports' / 'figures' / name
    if p.exists():
        display(Image(filename=str(p)))

### Reading the segments

The misalignment between k-means (which minimises within-cluster variance on
standardised features) and the realised default rate is the honest result here:
the largest variance directions in borrower characteristics are not the same as
the credit-risk directions. The segments are therefore useful for
**portfolio description and marketing/limit strategies**, while default risk
is carried by the supervised scorecard in notebook 03.

### Implementation notes

* Features are median-imputed and standardised; the median and scaler are fit
  on the fitting subsample only.
* k-means is fit on a subsample and then used to assign every loan, so the
  pipeline stays fast on the full sample.